# 01 — Data Preprocessing and Cleaning

Loads raw GPS/road-network trips, validates them, cleans them and writes `data/processed/cleaned/` plus the cleaning audit. **Run this first.**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries imported successfully.")

In [ ]:
working_directory = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        candidate
        for candidate in [working_directory, *working_directory.parents]
        if (candidate / "data" / "raw" / "train").is_dir()
        and (candidate / "data" / "raw" / "student_test").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate a project root containing data/raw/train "
        "and data/raw/student_test from the current working directory."
    )

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
TRAIN_DIR = RAW_DIR / "train"
TEST_DIR = RAW_DIR / "student_test"
PROCESSED_DIR = DATA_DIR / "processed"
CLEANED_DIR = PROCESSED_DIR / "cleaned"
FEATURE_DIR = PROCESSED_DIR / "features"
TRAIN_CLEANED_DIR = CLEANED_DIR / "train"
TEST_CLEANED_DIR = CLEANED_DIR / "student_test"
TRAIN_FEATURE_DIR = FEATURE_DIR / "train"
TEST_FEATURE_DIR = FEATURE_DIR / "student_test"
RESULTS_DIR = PROJECT_ROOT / "results"
PREPROCESSING_RESULTS_DIR = RESULTS_DIR / "preprocessing"

for directory in [
    TRAIN_CLEANED_DIR,
    TEST_CLEANED_DIR,
    TRAIN_FEATURE_DIR,
    TEST_FEATURE_DIR,
    PREPROCESSING_RESULTS_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Training directory:", TRAIN_DIR)
print("Student test directory:", TEST_DIR)
print("Processed directory:", PROCESSED_DIR)

In [ ]:
assert PROJECT_ROOT.exists(), "Project root does not exist."
assert TRAIN_DIR.exists(), "Training directory not found."
assert TEST_DIR.exists(), "Student test directory not found."

print("All required input directories exist.")


In [ ]:
def get_trip_directories(directory):

    return sorted(
        [
            folder
            for folder in directory.iterdir()
            if folder.is_dir()
        ]
    )


train_trips = get_trip_directories(TRAIN_DIR)
test_trips = get_trip_directories(TEST_DIR)

print("Training trips:", len(train_trips))
print("Student test trips:", len(test_trips))

print("\nSample training folders:")
for folder in train_trips[:5]:
    print(folder.name)

print("\nSample student test folders:")
for folder in test_trips[:5]:
    print(folder.name)
    

In [ ]:
def _find_single_file(trip_folder, extension):
    matches = sorted(Path(trip_folder).glob(f"*.{extension}"))
    if len(matches) != 1:
        raise ValueError(
            f"Expected exactly one .{extension} file in {trip_folder}, "
            f"found {len(matches)}."
        )
    return matches[0]


def load_trip(trip_folder):
    trip_folder = Path(trip_folder)
    if not trip_folder.is_dir():
        raise FileNotFoundError(f"Trip folder does not exist: {trip_folder}")

    tables = {
        "track": ("track", ["longitude", "latitude", "timestamp"]),
        "nodes": ("nodes", ["longitude", "latitude"]),
        "arcs": ("arcs", ["from_node", "to_node"]),
        "route": ("route", ["arc_id"]),
    }
    loaded = {}
    for key, (extension, columns) in tables.items():
        file_path = _find_single_file(trip_folder, extension)
        frame = pd.read_csv(
            file_path,
            sep=r"\s+",
            header=None,
            names=columns,
            on_bad_lines="error",
        )
        if frame.empty:
            raise ValueError(f"Input file is empty: {file_path}")
        loaded[key] = frame

    loaded["track"]["observation_id"] = np.arange(len(loaded["track"]))
    loaded["nodes"]["node_id"] = np.arange(len(loaded["nodes"]))
    loaded["arcs"]["arc_id"] = np.arange(len(loaded["arcs"]))
    return {"trip_id": trip_folder.name, **loaded}


EARTH_RADIUS_M = 6_371_000.0
TIMESTAMP_TO_SECONDS = 1.0
MAX_SPEED_MPS = 55.0
SPIKE_BRIDGE_SPEED_RATIO = 0.5


def haversine_distance(lon1, lat1, lon2, lat2):
    lon1 = np.radians(np.asarray(lon1, dtype=float))
    lat1 = np.radians(np.asarray(lat1, dtype=float))
    lon2 = np.radians(np.asarray(lon2, dtype=float))
    lat2 = np.radians(np.asarray(lat2, dtype=float))
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )
    a = np.clip(a, 0, 1)
    return EARTH_RADIUS_M * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))


def detect_gps_anomalies(
    track,
    timestamp_to_seconds=TIMESTAMP_TO_SECONDS,
    speed_threshold_mps=MAX_SPEED_MPS,
    bridge_speed_ratio=SPIKE_BRIDGE_SPEED_RATIO,
):
    required = {"observation_id", "longitude", "latitude", "timestamp"}
    missing = required.difference(track.columns)
    if missing:
        raise ValueError(f"Track is missing required columns: {sorted(missing)}")

    df = track.copy().sort_values("observation_id", kind="stable").reset_index(drop=True)
    for column in ["longitude", "latitude", "timestamp"]:
        df[column] = pd.to_numeric(df[column], errors="coerce")

    count = len(df)
    timestamp = df["timestamp"].to_numpy(dtype=float)
    longitude = df["longitude"].to_numpy(dtype=float)
    latitude = df["latitude"].to_numpy(dtype=float)
    time_raw = np.diff(timestamp)
    time_seconds = time_raw * timestamp_to_seconds
    distances = np.full(max(count - 1, 0), np.nan)
    speeds = np.full(max(count - 1, 0), np.nan)

    if count > 1:
        distances = haversine_distance(
            longitude[:-1], latitude[:-1], longitude[1:], latitude[1:]
        )
        valid = np.isfinite(distances) & np.isfinite(time_seconds) & (time_seconds > 0)
        np.divide(distances, time_seconds, out=speeds, where=valid)

    df["distance_from_previous_m"] = np.r_[np.nan, distances]
    df["time_difference_raw"] = np.r_[np.nan, time_raw]
    df["time_difference_s"] = np.r_[np.nan, time_seconds]
    df["speed_mps"] = np.r_[np.nan, speeds]
    df["speed_kmh"] = df["speed_mps"] * 3.6
    df["above_speed_threshold"] = df["speed_mps"].gt(speed_threshold_mps)

    isolated_spikes = np.zeros(count, dtype=bool)
    if count >= 3:
        bridge_distance = haversine_distance(
            longitude[:-2], latitude[:-2], longitude[2:], latitude[2:]
        )
        bridge_time = (timestamp[2:] - timestamp[:-2]) * timestamp_to_seconds
        bridge_speed = np.full(count - 2, np.nan)
        valid_bridge = (
            np.isfinite(bridge_distance)
            & np.isfinite(bridge_time)
            & (bridge_time > 0)
        )
        np.divide(bridge_distance, bridge_time, out=bridge_speed, where=valid_bridge)
        incoming_speed = speeds[:-1]
        outgoing_speed = speeds[1:]
        isolated_spikes[1:-1] = (
            (incoming_speed > speed_threshold_mps)
            & (outgoing_speed > speed_threshold_mps)
            & (
                bridge_speed
                <= bridge_speed_ratio * np.minimum(incoming_speed, outgoing_speed)
            )
        )

    df["possible_gps_outlier"] = isolated_spikes
    return df


print("Trip loader and GPS diagnostics defined.")

In [ ]:
sample_data = None
for trip_folder in train_trips:
    candidate = load_trip(trip_folder)
    if detect_gps_anomalies(candidate["track"])["possible_gps_outlier"].any():
        sample_data = candidate
        break

if sample_data is None:
    sample_data = load_trip(train_trips[0])

SAMPLE_TRIP_ID = sample_data["trip_id"]
track = sample_data["track"]
nodes = sample_data["nodes"]
arcs = sample_data["arcs"]
route = sample_data["route"]

print("Loaded sample trip:", SAMPLE_TRIP_ID)

In [ ]:
print("GPS TRACK")
display(track.head())

print("\nROAD NETWORK NODES")
display(nodes.head())

print("\nROAD NETWORK ARCS")
display(arcs.head())

print("\nREFERENCE ROUTE")
display(route.head())


In [ ]:
datasets = {
    "TRACK": track,
    "NODES": nodes,
    "ARCS": arcs,
    "ROUTE": route
}

for name, dataframe in datasets.items():

    print(f"\n{name}")
    print("-" * 40)

    print("Shape:", dataframe.shape)
    print("Data types:")

    display(dataframe.dtypes)
    

In [ ]:
for name, dataframe in datasets.items():

    print(f"\n{name} missing values")

    display(
        dataframe.isnull().sum().to_frame(
            name="missing_count"
        )
    )
    

In [ ]:
print(
    "Duplicate GPS coordinate-timestamp rows:",
    track.duplicated(
        subset=["longitude", "latitude", "timestamp"]
    ).sum()
)

print(
    "Duplicate road nodes:",
    nodes.duplicated(
        subset=["longitude", "latitude"]
    ).sum()
)

print(
    "Duplicate road arcs:",
    arcs.duplicated(
        subset=["from_node", "to_node"]
    ).sum()
)


In [ ]:
print("GPS longitude range:")
print(track["longitude"].min(), track["longitude"].max())

print("\nGPS latitude range:")
print(track["latitude"].min(), track["latitude"].max())

print("\nRoad-node longitude range:")
print(nodes["longitude"].min(), nodes["longitude"].max())

print("\nRoad-node latitude range:")
print(nodes["latitude"].min(), nodes["latitude"].max())


In [ ]:
timestamp_difference = track["timestamp"].diff()

print("Timestamp statistics:")
display(timestamp_difference.describe())

print("\nFirst timestamp values:")
display(track["timestamp"].head(10))

print("\nNon-positive intervals:",
      (timestamp_difference <= 0).sum())

print("\nIntervals greater than 60:",
      (timestamp_difference > 60).sum())
      

In [ ]:
plt.figure(figsize=(10, 7))

plt.plot(
    track["longitude"],
    track["latitude"],
    marker=".",
    markersize=3,
    linewidth=1
)

plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title(f"Raw GPS Trajectory — {SAMPLE_TRIP_ID}")

plt.grid(True)
plt.axis("equal")
plt.show()

In [ ]:
valid_node_ids = set(nodes["node_id"])

invalid_arc_references = (
    ~arcs["from_node"].isin(valid_node_ids)
    | ~arcs["to_node"].isin(valid_node_ids)
)

print("Invalid arc references:",
      invalid_arc_references.sum())

valid_arc_ids = set(arcs["arc_id"])

invalid_route_references = (
    ~route["arc_id"].isin(valid_arc_ids)
)

print("Invalid route references:",
      invalid_route_references.sum())
      

In [ ]:
def create_trip_inventory(trip_folders, partition):

    records = []

    for folder in trip_folders:

        data = load_trip(folder)

        records.append({
            "trip_id": data["trip_id"],
            "partition": partition,
            "gps_observations": len(data["track"]),
            "road_nodes": len(data["nodes"]),
            "road_arcs": len(data["arcs"]),
            "route_length": len(data["route"])
        })

    return records


inventory_records = []

inventory_records.extend(
    create_trip_inventory(train_trips, "train")
)

inventory_records.extend(
    create_trip_inventory(test_trips, "student_test")
)

dataset_inventory = pd.DataFrame(inventory_records)

display(dataset_inventory.head(10))

print("\nDataset summary:")
display(
    dataset_inventory.groupby("partition").agg(
        {
            "gps_observations": ["count", "mean", "min", "max"],
            "road_nodes": "mean",
            "road_arcs": "mean",
            "route_length": "mean"
        }
    )
)

dataset_inventory.to_csv(
    PREPROCESSING_RESULTS_DIR / "dataset_inventory.csv",
    index=False
)


DATA CLEANING AND VALIDATION

In [ ]:
def clean_trip(data):
    track = data["track"].copy()
    nodes = data["nodes"].copy()
    arcs = data["arcs"].copy()
    route = data["route"].copy()

    audit = {
        "trip_id": data["trip_id"],
        "original_gps": len(track),
        "original_nodes": len(nodes),
        "original_arcs": len(arcs),
        "original_route": len(route),
    }

    for column in ["longitude", "latitude", "timestamp"]:
        track[column] = pd.to_numeric(track[column], errors="coerce")
    for column in ["longitude", "latitude"]:
        nodes[column] = pd.to_numeric(nodes[column], errors="coerce")
    for column in ["from_node", "to_node", "arc_id"]:
        arcs[column] = pd.to_numeric(arcs[column], errors="coerce")
    route["arc_id"] = pd.to_numeric(route["arc_id"], errors="coerce")

    track = track.replace([np.inf, -np.inf], np.nan)
    nodes = nodes.replace([np.inf, -np.inf], np.nan)
    arcs = arcs.replace([np.inf, -np.inf], np.nan)
    route = route.replace([np.inf, -np.inf], np.nan)

    missing_gps = track[["longitude", "latitude", "timestamp"]].isna().any(axis=1)
    audit["gps_missing_or_nonfinite_removed"] = int(missing_gps.sum())
    track = track.loc[~missing_gps].copy()

    invalid_coordinates = (
        ~track["longitude"].between(-180, 180)
        | ~track["latitude"].between(-90, 90)
    )
    audit["invalid_coordinates_removed"] = int(invalid_coordinates.sum())
    track = track.loc[~invalid_coordinates].copy()

    duplicate_gps = track.duplicated(
        subset=["longitude", "latitude", "timestamp"], keep="first"
    )
    audit["duplicate_gps_removed"] = int(duplicate_gps.sum())
    track = track.loc[~duplicate_gps].copy()
    track = track.sort_values("observation_id", kind="stable").reset_index(drop=True)

    time_difference = track["timestamp"].diff()
    audit["non_increasing_intervals"] = int((time_difference <= 0).sum())
    audit["large_intervals_over_60_raw_units"] = int(
        (time_difference > 60).sum()
    )

    diagnostics = detect_gps_anomalies(track)
    spike_ids = set(
        diagnostics.loc[
            diagnostics["possible_gps_outlier"], "observation_id"
        ]
    )
    spike_mask = track["observation_id"].isin(spike_ids)
    audit["isolated_gps_spikes_removed"] = int(spike_mask.sum())
    track = track.loc[~spike_mask].copy().reset_index(drop=True)

    valid_node_coordinates = (
        nodes["longitude"].between(-180, 180)
        & nodes["latitude"].between(-90, 90)
    )
    nodes["valid_coordinates"] = valid_node_coordinates
    audit["invalid_nodes_retained_for_id_stability"] = int(
        (~valid_node_coordinates).sum()
    )
    valid_node_ids = set(nodes.loc[valid_node_coordinates, "node_id"].astype(int))

    invalid_arcs = (
        arcs["arc_id"].isna()
        | arcs["from_node"].isna()
        | arcs["to_node"].isna()
        | ~arcs["from_node"].isin(valid_node_ids)
        | ~arcs["to_node"].isin(valid_node_ids)
    )
    audit["invalid_arcs_removed"] = int(invalid_arcs.sum())
    arcs = arcs.loc[~invalid_arcs].copy()
    arcs[["from_node", "to_node", "arc_id"]] = arcs[
        ["from_node", "to_node", "arc_id"]
    ].astype(int)

    original_arc_ids = set(
        pd.to_numeric(data["arcs"]["arc_id"], errors="coerce")
        .dropna()
        .astype(int)
    )
    cleaned_arc_ids = set(arcs["arc_id"])
    invalid_route = (
        route["arc_id"].isna()
        | ~route["arc_id"].isin(original_arc_ids)
        | ~route["arc_id"].isin(cleaned_arc_ids)
    )
    audit["invalid_route_references_removed"] = int(invalid_route.sum())
    route = route.loc[~invalid_route].copy()
    route["arc_id"] = route["arc_id"].astype(int)

    audit["cleaned_gps"] = len(track)
    audit["cleaned_nodes"] = len(nodes)
    audit["cleaned_arcs"] = len(arcs)
    audit["cleaned_route"] = len(route)
    cleaned_data = {
        "trip_id": data["trip_id"],
        "track": track,
        "nodes": nodes,
        "arcs": arcs,
        "route": route,
    }
    return cleaned_data, audit

In [ ]:
cleaned_sample, sample_audit = clean_trip(sample_data)

print("Cleaning audit:")
display(pd.Series(sample_audit))

print("\nCleaned GPS:")
display(cleaned_sample["track"].head())

print("\nCleaned nodes:")
display(cleaned_sample["nodes"].head())

print("\nCleaned arcs:")
display(cleaned_sample["arcs"].head())

print("\nCleaned route:")
display(cleaned_sample["route"].head())

In [ ]:
sample_anomalies = detect_gps_anomalies(sample_data["track"])
candidate_points = sample_anomalies.loc[
    sample_anomalies["possible_gps_outlier"]
]
removed_ids = set(sample_data["track"]["observation_id"]) - set(
    cleaned_sample["track"]["observation_id"]
)
removed_points = sample_data["track"].loc[
    sample_data["track"]["observation_id"].isin(removed_ids)
]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes[0].plot(
    sample_anomalies["observation_id"],
    sample_anomalies["speed_kmh"],
    color="steelblue",
    linewidth=1,
    label="Raw GPS speed",
)
axes[0].axhline(
    MAX_SPEED_MPS * 3.6,
    color="black",
    linestyle="--",
    label=f"Screening threshold ({MAX_SPEED_MPS * 3.6:.0f} km/h)",
)
if not candidate_points.empty:
    axes[0].scatter(
        candidate_points["observation_id"],
        candidate_points["speed_kmh"],
        color="red",
        zorder=3,
        label="Isolated spike removed",
    )
axes[0].set_xlabel("Observation ID")
axes[0].set_ylabel("Speed (km/h)")
axes[0].set_title("GPS speed screening")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(
    sample_data["track"]["longitude"],
    sample_data["track"]["latitude"],
    color="lightgray",
    linewidth=1,
    label="Original track",
)
axes[1].plot(
    cleaned_sample["track"]["longitude"],
    cleaned_sample["track"]["latitude"],
    color="green",
    linewidth=1,
    label="Cleaned track",
)
if not removed_points.empty:
    axes[1].scatter(
        removed_points["longitude"],
        removed_points["latitude"],
        color="red",
        marker="x",
        s=70,
        label="Removed spike",
        zorder=4,
    )
axes[1].set_xlabel("Longitude")
axes[1].set_ylabel("Latitude")
axes[1].set_title(f"Original vs cleaned GPS: {SAMPLE_TRIP_ID}")
axes[1].set_aspect("equal", adjustable="datalim")
axes[1].grid(True)
axes[1].legend()

plt.tight_layout()
plt.show()
print("Isolated spikes removed:", len(removed_points))

In [ ]:
def save_cleaned_trip(data, output_directory):

    trip_output = output_directory / data["trip_id"]

    trip_output.mkdir(
        parents=True,
        exist_ok=True
    )

    data["track"].to_csv(
        trip_output / "track.csv",
        index=False
    )

    data["nodes"].to_csv(
        trip_output / "nodes.csv",
        index=False
    )

    data["arcs"].to_csv(
        trip_output / "arcs.csv",
        index=False
    )

    data["route"].to_csv(
        trip_output / "route.csv",
        index=False
    )


print("Cleaned-data saving function defined.")


In [ ]:
cleaning_audits = []

for trip_folder in train_trips:

    data = load_trip(trip_folder)

    cleaned_data, audit = clean_trip(data)

    save_cleaned_trip(
        cleaned_data,
        TRAIN_CLEANED_DIR
    )

    audit["partition"] = "train"

    cleaning_audits.append(audit)


for trip_folder in test_trips:

    data = load_trip(trip_folder)

    cleaned_data, audit = clean_trip(data)

    save_cleaned_trip(
        cleaned_data,
        TEST_CLEANED_DIR
    )

    audit["partition"] = "student_test"

    cleaning_audits.append(audit)


cleaning_audit_df = pd.DataFrame(cleaning_audits)

display(cleaning_audit_df.head())

print("Trips cleaned:", len(cleaning_audit_df))


In [ ]:
cleaning_audit_df.to_csv(
    PREPROCESSING_RESULTS_DIR / "cleaning_audit.csv",
    index=False
)

print("Cleaning audit saved.")


In [ ]:
print("Cleaning audit columns:")
print(cleaning_audit_df.columns.tolist())

print("\nCleaning audit:")
display(cleaning_audit_df.head(10))

print("\nTotal cleaning audit:")
display(cleaning_audit_df.sum(numeric_only=True))

In [ ]:
print("Timestamp statistics:")

display(
    cleaning_audit_df[
        ["large_intervals_over_60_raw_units"]
    ].describe()
)

print("\nTrips with large timestamp intervals:")

display(
    cleaning_audit_df[
        cleaning_audit_df["large_intervals_over_60_raw_units"] > 0
    ]
)


In [ ]:
print("Timestamp-gap audit (raw timestamp units):")
display(
    cleaning_audit_df["large_intervals_over_60_raw_units"].describe().to_frame()
)

print("Trips with gaps greater than 60 raw units:")
display(
    cleaning_audit_df.loc[
        cleaning_audit_df["large_intervals_over_60_raw_units"] > 0,
        ["trip_id", "large_intervals_over_60_raw_units", "partition"],
    ]
)

print(
    "These gaps are reported, not deleted: a long pause/gap alone "
    "does not prove a GPS fix is invalid."
)

In [ ]:
total_before = int(cleaning_audit_df["original_gps"].sum())
total_after = int(cleaning_audit_df["cleaned_gps"].sum())
removed = total_before - total_after

print("Total GPS observations before:", f"{total_before:,}")
print("Total GPS observations after:", f"{total_after:,}")
print("Observations removed:", f"{removed:,}")
print(
    "Percentage removed:",
    round(removed / total_before * 100, 4) if total_before else 0,
    "%",
)

changed_trips = cleaning_audit_df.loc[
    cleaning_audit_df["original_gps"] != cleaning_audit_df["cleaned_gps"],
    [
        "trip_id",
        "original_gps",
        "cleaned_gps",
        "isolated_gps_spikes_removed",
        "partition",
    ],
]
print("Trips with GPS rows removed:", len(changed_trips))
display(changed_trips)

In [ ]:
expected_trip_ids = {folder.name for folder in train_trips + test_trips}
assert cleaning_audit_df["trip_id"].is_unique
assert set(cleaning_audit_df["trip_id"]) == expected_trip_ids
assert len(cleaning_audit_df) == len(expected_trip_ids)

for before_column, after_column in [
    ("original_gps", "cleaned_gps"),
    ("original_nodes", "cleaned_nodes"),
    ("original_arcs", "cleaned_arcs"),
    ("original_route", "cleaned_route"),
]:
    assert (cleaning_audit_df[after_column] <= cleaning_audit_df[before_column]).all()

clean_track = cleaned_sample["track"]
assert clean_track["observation_id"].is_unique
assert not clean_track.duplicated(
    subset=["longitude", "latitude", "timestamp"]
).any()
assert clean_track["longitude"].between(-180, 180).all()
assert clean_track["latitude"].between(-90, 90).all()
assert sample_audit["isolated_gps_spikes_removed"] == int(
    sample_anomalies["possible_gps_outlier"].sum()
)

print("Validation passed.")
print("Trips loaded and audited:", len(cleaning_audit_df))
print("GPS rows removed:", int(removed))
print(
    "Isolated GPS spikes removed:",
    int(cleaning_audit_df["isolated_gps_spikes_removed"].sum()),
)

In [ ]:
gps_counts = cleaning_audit_df[["original_gps", "cleaned_gps"]].to_numpy().ravel()
bin_edges = np.histogram_bin_edges(gps_counts, bins=20)
removed_per_trip = cleaning_audit_df["original_gps"] - cleaning_audit_df["cleaned_gps"]
changed = cleaning_audit_df.loc[removed_per_trip > 0].copy()
changed["gps_removed"] = removed_per_trip.loc[removed_per_trip > 0]

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
sns.histplot(cleaning_audit_df["original_gps"], bins=bin_edges, color="steelblue", alpha=0.55, ax=axes[0])
axes[0].set_title("GPS observations before cleaning")
axes[0].set_xlabel("Observations per trip")

sns.histplot(cleaning_audit_df["cleaned_gps"], bins=bin_edges, color="seagreen", alpha=0.55, ax=axes[1])
axes[1].set_title("GPS observations after cleaning")
axes[1].set_xlabel("Observations per trip")

if changed.empty:
    axes[2].text(0.5, 0.5, "No GPS rows removed", ha="center", va="center")
    axes[2].set_axis_off()
else:
    axes[2].bar(changed["trip_id"], changed["gps_removed"], color="tomato")
    axes[2].set_title("GPS rows removed per trip")
    axes[2].set_xlabel("Trip ID")
    axes[2].set_ylabel("Rows removed")
    axes[2].tick_params(axis="x", rotation=70)

plt.tight_layout()
plt.show()